# 05 · Agent tool evaluation
**Objectives:** inspect an explicit Converse tool loop, verify completed calls and arguments, and evaluate final-answer quality separately.

The nine original Dungeon Master scenarios and all game fixtures are preserved. Four local tools read inventory, read character stats, describe scenes, and roll dice. Each scenario starts fresh; no game state persists between requests. Seeded dice make offline tool tests repeatable.

Live workload: up to **9 × 20 Converse requests plus 7 rubric requests**, excluding retries. The loop limit bounds requests per scenario; it does not cap dollars. The configured model must support Converse tool use.


In [ ]:
from pathlib import Path
import json
import os
import sys

# Works from labs/boto3 or its notebooks directory, including a fresh clone.
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / "pyproject.toml").is_file() and (p / "data/synthetic.json").is_file())
sys.path.insert(0, str(ROOT / "src"))
from dotenv import load_dotenv
load_dotenv(ROOT / ".env", override=False)
from ai_eval_labs.core import Bedrock, DATA, Result, export_results, generate, load_cases
from ai_eval_labs.metrics import Judge, check, contains

LIVE = False  # Set True only after reviewing AWS model access, prices, and request counts.
api = Bedrock(live=LIVE)
judge = Judge(api)
synthetic = json.loads((DATA / "synthetic.json").read_text())
print("LIVE: paid Bedrock calls" if LIVE else "OFFLINE: synthetic teaching fixtures; no AWS calls")


In [ ]:
from ai_eval_labs.agent import GameTools, ScriptedAgent, run_agent, verify_tools
from ai_eval_labs.lessons import evaluate_agent
cases = load_cases("agents")
[(case["id"], case["description"]) for case in cases]


## Worked example: dispatch before grading
For each assistant tool request, the loop validates the registered tool name and arguments, executes it locally, then sends a `toolResult` with the matching `toolUseId`. Only successful executions enter `tool_calls`. Unknown tools and invalid arguments produce error results and are never counted as completed evidence.

Offline mode runs the real tool dispatcher against scripted assistant messages. Its final prose is hand-authored and independent of the seeded tool results; it demonstrates assertion plumbing and cannot establish that a model used a result correctly.


In [ ]:
tools = GameTools(seed=42)
tools.execute("roll_dice", {"sides": 20, "count": 1, "modifier": 3, "purpose": "ability check"})


In [ ]:
results = []
for index, case in enumerate(cases):
    transport = api if LIVE else ScriptedAgent(index, synthetic["agents"][index])
    result = run_agent(transport, case["id"], case["query"], max_turns=20, seed=None if LIVE else 42)
    evaluate_agent(result, case, index, judge)
    results.append(result)
[(r.scenario_id, r.tool_calls, r.assertions, r.errors) for r in results]


## Negative control: mentioning a tool is not execution
The attack scenarios require an executed d20 roll with count 1. Stats and inventory require `player1`; the crypt scenario requires location `crypt`. The short-rest case must not roll dice before the player agrees to spend Hit Dice. Grading checks selected calls and arguments, not every D&D mechanic or the order of all actions.


In [ ]:
fake = Result("negative-control", "synthetic-agent", response="I used roll_dice with sides 20 and count 1.")
assert verify_tools(fake, 0)["passed"] is False
print(verify_tools(fake, 0))
export_results(results, ROOT / "results/05_agent")


## Exercise
Change a scripted tool request to use `player2`, or set the dice count to zero. Inspect the error result and confirm it is absent from completed calls. Create a scripted short-rest response that calls `roll_dice`; verify that the short-rest check fails.

**Hint:** `ScriptedAgent.requests` captures requests; a custom object with a `converse(**request)` method can return your own message sequence. Inspect `run_agent` in `src/ai_eval_labs/agent.py` to see the full loop.

**Checkpoints:** Does a correct tool call establish correct use of its value? Which errors invalidate a run? How would persistent state change isolation between evaluations? Why must a random attack not be graded as a guaranteed critical hit?

Return to the [course guide](../README.md).
